<a href="https://colab.research.google.com/github/n-schiozer/met_comp_economistas_unesp/blob/main/aula5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos computacionais para economistas
## Aula 5 — Pandas I

**Pergunta da aula:** como importar uma planilha com dados do Brasil e obter informações econômicas usando Python?

**Tópicos:**

- Entender o que são `pandas` e um `DataFrame`.
- Ler a aba `Dados` de um arquivo Excel com `pd.read_excel()`.
- Inspecionar a base: linhas, colunas, dimensões e tipos de variáveis.
- Selecionar colunas, filtrar observações e combinar condições.
- Calcular estatísticas simples e criar uma coluna de crescimento nominal do PIB **com uma função criada pelos alunos**.
- Descobrir o método `pct_change()` do Pandas e comparar as duas soluções.
- Descobrir métodos e consultar a ajuda do Pandas com autocompletar, `dir()` e `help()`; exportar um resultado para Excel.



## 1. O que muda ao trabalhar com dados reais?

Nas aulas anteriores usamos um **dicionário aninhado** com poucos anos. No trabalho com dados econômicos, é comum receber uma **planilha** com uma linha por ano e uma coluna por indicador.

O **Pandas** é uma biblioteca que permite ler essas planilhas e trabalhar com tabelas usando Python. Sua estrutura de tabela principal se chama **`DataFrame`**.

**Como ler nossa planilha:**

| Coluna do Excel | Conteúdo | Unidade |
|---|---|---|
| `ano` | Ano da observação | Ano civil |
| `pib_nominal_bilhoes_rs` | PIB a preços correntes | R$ bilhões |
| `pib_crescimento_real_pct` | Crescimento real do PIB | % ao ano |
| `desemprego_pct` | Taxa anual de desocupação | % |
| `inflacao_ipca_pct` | IPCA acumulado no ano | % ao ano |

**Atenção às unidades:** o valor `4.26` da inflação corresponde a **4,26%**, não a 426%. Ele já está expresso em pontos percentuais na tabela; não devemos multiplicá-lo novamente por 100.

### Preparação — disponibilizar o Excel

O notebook precisa encontrar o arquivo `dados_macro_brasil.xlsx`. **No Colab**, a célula abaixo abre o seletor para enviar a planilha caso ela ainda não esteja disponível. **No computador**, basta colocar o Excel na mesma pasta do notebook.

A parte de identificação e envio do arquivo é apenas uma preparação para a aula; o conteúdo novo começa em `pd.read_excel()`.

In [27]:
import pandas as pd
from pathlib import Path

arquivo = "dados_macro_brasil.xlsx"

if not Path(arquivo).exists():
    try:
        from google.colab import files
        print("Selecione a planilha Excel disponibilizada para a aula.")
        files.upload()
    except ImportError:
        print("Coloque a planilha Excel na mesma pasta do notebook.")

if not Path(arquivo).exists():
    raise FileNotFoundError(f"Não encontrei: {arquivo}")

print("Planilha localizada:", arquivo)

Planilha localizada: dados_macro_brasil.xlsx


### Importação: Excel → DataFrame

- `pd` é o apelido da biblioteca `pandas`.
- `read_excel()` é a função que **lê** o arquivo.
- `sheet_name="Dados"` escolhe a aba correta.
- `dados` é o nome da variável que guarda nosso `DataFrame`.
- `head()` exibe as primeiras linhas para verificar se a leitura funcionou.

In [ ]:
import pandas as pd # Importação de pacote.

In [28]:
dados = pd.read_excel(arquivo)
dados.head()

,Data,PIB per capita,PIB - preços de mercado,PIB - preço 2010,Desemprego,IPCA
0,1991,0.000411,60.285999,2.114277e+06,6.366,472.70
1,1992,0.004311,640.958768,2.102773e+06,6.420,1119.10
2,1993,0.093688,14097.114182,2.206329e+06,6.030,2477.15
3,1994,2.291786,349204.679000,2.335462e+06,6.470,916.46
4,1995,4.568805,705991.552861,2.434107e+06,7.090,22.41


**Pergunta:** o que é uma **linha** da tabela? O que representa cada **coluna**?

**Obs:** há uma linha para cada ano (2012–2025), e não para cada mês. O IPCA é acumulado **no ano**, enquanto o desemprego é uma estimativa **anual** da PNAD Contínua.

## 2. Inspecionar o DataFrame

Antes de calcular qualquer coisa, um economista deve conhecer a estrutura do conjunto de dados.

| Comando | O que informa |
|---|---|
| `dados.head(3)` | Três primeiras linhas |
| `dados.tail(3)` | Três últimas linhas |
| `dados.shape` | Quantidade de linhas e colunas |
| `dados.columns` | Nomes das colunas |
| `dados.dtypes` | Tipo dos valores em cada coluna |
| `dados.info()` | Estrutura e presença de valores não ausentes |

In [29]:
print("Dimensões (linhas, colunas):", dados.shape)
print("Colunas:", list(dados.columns))
print("\nÚltimas observações:")
dados.tail(3)


Dimensões (linhas, colunas): (34, 6)
Colunas: ['Data', 'PIB per capita', 'PIB - preços de mercado ', 'PIB - preço 2010', 'Desemprego', 'IPCA']

Últimas observações:


,Data,PIB per capita,PIB - preços de mercado,PIB - preço 2010,Desemprego,IPCA
31,2022,49.63383,1.007968e+07,4.333844e+06,9.250,5.79
32,2023,53.64454,1.094334e+07,4.474332e+06,7.975,4.62
33,2024,57.32999,1.177925e+07,4.626275e+06,6.925,4.83


In [30]:

dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Data                      34 non-null     int64  
 1   PIB per capita            34 non-null     float64
 2   PIB - preços de mercado   34 non-null     float64
 3   PIB - preço 2010          34 non-null     float64
 4   Desemprego                34 non-null     float64
 5   IPCA                      34 non-null     float64
dtypes: float64(5), int64(1)
memory usage: 1.7 KB


### Como interpretar a saída de `dados.info()`

O comando `dados.info()` apresenta um **resumo da estrutura da tabela**. Ele não calcula indicadores econômicos: mostra quantas observações existem, quais são as colunas, como os valores estão armazenados e se há dados ausentes.

| Informação exibida | Como interpretar |
|---|---|
| `<class 'pandas.core.frame.DataFrame'>` | Indica que `dados` é um objeto do tipo **DataFrame**, ou seja, uma tabela do Pandas. |
| `RangeIndex: 34 entries, 0 to 33` | Há **34 linhas (observações)**, identificadas pelos índices de **0 a 33**. O índice é a posição/identificação atribuída pelo Pandas e **não é necessariamente o ano**. |
| `Data columns (total 6 columns)` | A tabela possui **6 colunas (variáveis)**. |
| `#` | Número da posição da coluna, começando em zero. |
| `Column` | Nome de cada variável (`Data`, `PIB per capita`, `IPCA` etc., no exemplo). |
| `Non-Null Count` | Quantidade de observações **não ausentes** em cada variável. `34 non-null` significa que **nenhuma das 34 linhas está vazia** naquela coluna. |
| `Dtype` | Tipo de dado armazenado: `int64` representa números inteiros; `float64`, números numéricos que podem ter casas decimais; `object` geralmente representa textos ou tipos mistos. |
| `dtypes: float64(5), int64(1)` | Resumo: **cinco variáveis** do tipo `float64` e **uma** do tipo `int64`. |
| `memory usage: 1.7 KB` | Estimativa da memória usada para armazenar a estrutura da tabela; **não é o tamanho do arquivo Excel no disco**. |

**Lendo o exemplo acima:** são **34 observações × 6 variáveis**. Todas as seis colunas apresentam `34 non-null`, portanto não há valores reconhecidos pelo Pandas como ausentes nessa tabela.

## Acessar linhas e colunas por posição: `iloc`

Usamos `head()` e `tail()` para visualizar várias linhas. Para acessar uma **posição específica**, usamos `iloc` (*indexação por posição inteira*).

Como nas listas, a contagem começa em **zero**.

| Código | Resultado |
|---|---|
| `dados.iloc[0]` | Primeira **linha** inteira |
| `dados.iloc[-1]` | Última **linha** inteira |
| `dados.iloc[:, 0]` | Primeira **coluna** inteira |
| `dados.iloc[0, 0]` | Valor na primeira linha e primeira coluna |
| `dados.iloc[0, 1]` | Valor na primeira linha e segunda coluna |

Em `dados.iloc[linha, coluna]`, a vírgula separa a posição da linha e a da coluna. O símbolo `:` significa **todas as posições** naquela dimensão.

Também podemos escolher uma coluna pelo **nome**: `dados["ano"]`. Portanto, `dados["ano"].iloc[0]` recupera o **primeiro ano**, e `dados["ano"].iloc[-1]`, o **último ano**.

**Atenção:** `iloc` identifica **posições**, não anos ou nomes de colunas. Na seção seguinte vamos aprofundar a seleção de colunas pelo nome.

### Exercício 1 — conhecer a base

1. Exiba as **quatro primeiras linhas**.
2. Exiba as **duas últimas linhas**.
3. Descubra quantas observações e quantas colunas existem.
4. Identifique o ano inicial e o final da base.

**Antes de executar:** qual deve ser o formato (linhas, colunas)?

In [48]:
# Exiba as primeiras 4 linhas com head().
# Exiba as últimas 2 linhas com tail().
# Mostre shape.
# Consulte os anos da primeira e da última observação.




34


np.int64(2024)

## 3. Selecionar colunas — transformar perguntas em consultas

Uma coluna pode ser selecionada com `dados["nome_da_coluna"]`.

Duas ou mais colunas são selecionadas com **uma lista** de nomes: `dados[["coluna1", "coluna2"]]`.

Isso retoma a diferença entre um valor e uma lista vista nas aulas anteriores: o nome de **uma** coluna vai entre um par de colchetes, enquanto **várias** colunas exigem uma lista dentro dos colchetes.

In [51]:
print("Colunas:", list(dados.columns))

Colunas: ['Data', 'PIB per capita', 'PIB - preços de mercado ', 'PIB - preço 2010', 'Desemprego', 'IPCA']


In [50]:
# Uma coluna: a taxa de desemprego
print(dados["Desemprego"].head())

# Duas colunas: ano e desemprego
dados[["Data", "Desemprego"]].head()

0    6.366
1    6.420
2    6.030
3    6.470
4    7.090
Name: Desemprego, dtype: float64


,Data,Desemprego
0,1991,6.366
1,1992,6.420
2,1993,6.030
3,1994,6.470
4,1995,7.090


### `Series` e `DataFrame`

Uma única coluna selecionada com `dados["desemprego_pct"]` produz uma **`Series`** (uma sequência de valores com índice). Duas ou mais colunas produzem um **`DataFrame`**.

Não precisamos decorar esses nomes de imediato. Importa compreender o que estamos selecionando.

In [53]:
uma_coluna = dados["Desemprego"]
duas_colunas = dados[["Data", "Desemprego"]]

print("Uma coluna:", type(uma_coluna).__name__)
print("Duas colunas:", type(duas_colunas).__name__)

Uma coluna: Series
Duas colunas: DataFrame


### Exercício 2 — inflação e crescimento real

1. Selecione apenas as colunas `Data` e `IPCA`.
2. Exiba as **cinco observações mais recentes** dessa seleção.


In [ ]:
# Selecione ano e inflação; depois use tail(5).


## 4. Filtrar observações — recuperar os anos que interessam

Uma comparação em uma coluna produz `True` ou `False` para **cada linha**. Podemos usar esses valores para manter apenas as observações que atendem ao critério.

A ideia é a mesma dos condicionais vistos antes, agora aplicada a uma tabela inteira.

In [56]:
# Anos em que a inflação esteve abaixo de 3 %
inflacao_baixa = dados[dados["IPCA"] < 3]=

print(inflacao_baixa)

    Data  PIB per capita  PIB - preços de mercado   PIB - preço 2010  \
7   1998        6.134541              1.002351e+06      2.581030e+06   
26  2017       32.474300              6.585479e+06      4.029775e+06   

    Desemprego  IPCA  
7        10.15  1.65  
26       12.85  2.95  


### Filtros por período

Também podemos filtrar pelo ano. Primeiro criamos a condição; depois a usamos para selecionar as linhas.

In [58]:
a_partir_2020 = dados["Data"] >= 2020

dados[a_partir_2020][["Data", "Desemprego", "IPCA"]]

,Data,Desemprego,IPCA
29,2020,13.775,4.52
30,2021,13.200,10.06
31,2022,9.250,5.79
32,2023,7.975,4.62
33,2024,6.925,4.83


### Combinar duas condições

Em tabelas Pandas, `&` representa **e** (ambas as condições devem ser verdadeiras), e `|` representa **ou**.

**Atenção:** precisamos colocar cada comparação entre parênteses.

Esta é uma diferença em relação a `and` e `or` com valores booleanos simples da Aula 3.

In [61]:
selecao = dados[
    (dados["Data"] >= 2020) &
    (dados["IPCA"] > 5)
]

print(selecao)

selecao[["Data", "IPCA"]]



    Data  PIB per capita  PIB - preços de mercado   PIB - preço 2010  \
30  2021        44.22367              9.012142e+06      4.206934e+06   
31  2022        49.63383              1.007968e+07      4.333844e+06   

    Desemprego   IPCA  
30       13.20  10.06  
31        9.25   5.79  


,Data,IPCA
30,2021,10.06
31,2022,5.79


### Exercício 3 — identificar períodos

1. Filtre os anos em que a **taxa de desemprego foi superior a 10%**.
2. Exiba apenas as colunas `ano` e `desemprego_pct`.
3. Depois, selecione os anos **a partir de 2020** com **inflação superior a 5%**.

**Pergunta:** qual é a diferença entre usar dois critérios com `&` e filtrar apenas um deles?

In [63]:
# Crie um filtro para desemprego superior a 10%.
# Selecione ano e desemprego.
# Crie outro filtro para ano >= 2020 e inflação > 5%.



,Data,PIB per capita,PIB - preços de mercado,PIB - preço 2010,Desemprego,IPCA
7,1998,6.134541,1.002351e+06,2.581030e+06,10.150,1.65
8,1999,6.524374,1.087710e+06,2.593107e+06,11.130,8.94
9,2000,7.061825,1.199092e+06,2.706892e+06,10.530,5.97
10,2001,7.629474,1.315755e+06,2.744515e+06,10.650,7.67
11,2002,8.520988,1.488787e+06,2.828317e+06,10.640,12.53
12,2003,9.722220,1.717950e+06,2.860584e+06,11.170,9.30
13,2004,10.966911,1.957751e+06,3.025352e+06,10.070,7.60
14,2005,12.041113,2.170585e+06,3.122228e+06,10.550,5.69
25,2016,31.078120,6.269327e+06,3.977162e+06,11.650,6.29
26,2017,32.474300,6.585479e+06,4.029775e+06,12.850,2.95


## 5. Resumir os dados — estatísticas simples

Pandas permite calcular estatísticas de uma coluna:

| Método | Interpretação |
|---|---|
| `.mean()` | Média aritmética |
| `.max()` | Maior valor |
| `.min()` | Menor valor |
| `.sum()` | Soma |

**Economia:** a média das **taxas anuais de inflação** é diferente da inflação **acumulada ao longo de todos os anos**. Não devemos somar ou tirar média de taxas sem entender a pergunta econômica.

In [64]:
media_ipca = dados["IPCA"].mean()
maior_ipca = dados["IPCA"].max()
menor_desemprego = dados["Desemprego"].min()

print(f"IPCA médio anual (2012–2025): {media_ipca:.2f}%")
print(f"Maior IPCA anual: {maior_ipca:.2f}%")
print(f"Menor taxa anual de desemprego: {menor_desemprego:.2f}%")

IPCA médio anual (2012–2025): 152.59%
Maior IPCA anual: 2477.15%
Menor taxa anual de desemprego: 6.03%


### Descobrir em qual ano ocorreu o valor máximo

`.idxmax()` devolve o **índice da linha** que contém o maior valor da coluna. `.loc[...]` permite recuperar essa linha da tabela.

Na nossa base, `ano` é uma **coluna**; o índice das linhas é criado automaticamente por Pandas.

In [65]:
indice_max = dados["IPCA"].idxmax()
dados.loc[indice_max, ["Data", "IPCA"]]

,2
Data,1993.00
IPCA,2477.15


### Exercício 4 — desemprego e inflação

1. Calcule a **taxa média de desemprego** em 2012–2025.
2. Encontre a **maior taxa de desemprego** no período.
3. Descubra **em qual ano** ela ocorreu, combinando `.idxmax()` e `.loc`.



In [ ]:
# Calcule a média de desemprego.
# Mostre sua taxa máxima.
# Use idxmax() e loc para encontrar o ano da taxa máxima.
# Compare esse ano com o ano de maior IPCA.

## 6. Crescimento nominal do PIB — primeiro vamos programar o cálculo

Nossa planilha contém o PIB **nominal**, em bilhões de reais, e o PIB **real** . Queremos construir uma coluna adicional: a variação percentual do PIB nominal de um ano para o seguinte.


**Desafio:** antes de conhecer o recurso pronto do Pandas, vamos resolver o problema utilizando **funções, listas e laços**. Isso ajuda a compreender o que acontece quando calculamos a variação de uma série inteira.

**Interpretação econômica:** crescimento nominal incorpora mudanças de quantidades *e* preços. Crescimento real busca medir a variação de volume. O IPCA **não** é o deflator do PIB.

In [ ]:
# Recuperar os valores do PIB nominal na forma de lista.
# Assim, podemos usar as operações com listas que já conhecemos.
pib_nominal = dados["PIB"].tolist()
print("Primeiros quatro valores do PIB nominal:", pib_nominal[:4])

### Exercício 5 — construir nossa própria função de variação percentual

**Parte A — Função**

1. Defina uma função `calcular_variacao_pct(valor_anterior, valor_atual)` que **retorne** a variação percentual entre os dois valores.
2. Teste com `calcular_variacao_pct(100, 110)`. O resultado esperado é **aproximadamente 10%** (pode aparecer `10.000000000000009` por arredondamento do computador).

**Parte B — Aplicar a função à série do PIB nominal**

3. Crie uma lista chamada `taxas_manuais`, iniciando com `[float("nan")]`. Precisamos marcar o primeiro ano como ausente, pois a base não contém o PIB nominal do ano anterior.
4. Utilize `for i in range(1, len(pib_nominal))` para percorrer os valores a partir do segundo ano.
5. Em cada repetição, chame sua função com `pib_nominal[i - 1]` e `pib_nominal[i]`, adicionando o resultado a `taxas_manuais` com `.append()`.
6. Adicione os resultados à tabela como uma coluna chamada `crescimento_nominal_funcao_pct` e exiba as primeiras linhas.

**Pistas:** `return` devolve o resultado; `lista[i-1]` acessa o valor anterior; `lista[i]` acessa o valor atual. A lista das variações deve ter o **mesmo número de elementos** que a base.

**Pergunta:** por que não atribuímos crescimento igual a zero ao primeiro ano?

In [ ]:
# PARTE A — escreva e teste a função calcular_variacao_pct().


# PARTE B — crie taxas_manuais = [float("nan")].
# Utilize for i in range(1, len(pib_nominal)):
#     chame sua função com os valores anterior e atual;
#     adicione o resultado à lista com append().


# Crie a coluna dados["crescimento_nominal_funcao_pct"].
# Exiba ano, PIB nominal e sua nova coluna com head().

### O Pandas já possui esse cálculo: `pct_change()`

Depois de criar nosso próprio procedimento, podemos usar o método **`pct_change()`**, que calcula a variação relativa entre observações consecutivas:

**Atenção:** `pct_change()` devolve uma **proporção**. Para exibir a variação em **%**, precisamos multiplicar por `100`. Assim, 0,10 corresponde a 10%.

Isso evita escrever um `for` para percorrer cada ano, mas implementa a mesma ideia do nosso exercício.

In [ ]:
# Uma única operação com o método pronto do Pandas.
dados["pib_nominal_crescimento_pct"] = (
    dados["pib_nominal_bilhoes_rs"].pct_change() * 100
)

dados[["ano", "pib_nominal_bilhoes_rs", "pib_nominal_crescimento_pct"]].head(6)

### Conferir as duas soluções

O cálculo feito **pela função** e o cálculo com **`pct_change()`** devem ser iguais, salvo pequenas diferenças de arredondamento numérico.

Se você ainda não terminou o exercício, a próxima célula avisará que a coluna calculada manualmente está faltando. Quando terminar, execute-a novamente.

In [ ]:
if "crescimento_nominal_funcao_pct" in dados.columns:
    comparacao = dados[[
        "ano", "crescimento_nominal_funcao_pct", "pib_nominal_crescimento_pct"
    ]]
    print(comparacao)
    diferenca_maxima = (
        comparacao["crescimento_nominal_funcao_pct"]
        - comparacao["pib_nominal_crescimento_pct"]
    ).abs().max()
    print(f"Maior diferença entre os dois cálculos: {diferenca_maxima:.10f} p.p.")
else:
    print("Complete o Exercício 5 para comparar sua função com pct_change().")

### Por que o primeiro crescimento nominal ficou ausente (`NaN`)?

Para calcular a variação nominal de **2012**, precisaríamos do PIB nominal de **2011**, que não está na planilha. `NaN` representa um resultado numérico ausente ou não disponível.

Nossa solução manual começou com `float("nan")`, enquanto `pct_change()` já fornece o valor ausente no primeiro ano automaticamente.

O crescimento **real** de 2012 pode estar disponível porque essa coluna veio de uma série oficial que inclui observações anteriores às apresentadas na planilha.

**Não substitua esse valor por zero:** ausência de informação não é o mesmo que crescimento nulo.

### Discussão — como descobrir os métodos disponíveis no Pandas?

Não precisamos decorar todos os comandos do Pandas. Podemos **explorar o que um objeto oferece** e consultar a documentação quando precisarmos de uma operação nova.

- **Autocompletar no Colab:** digite `dados.` e pressione **Tab** para ver sugestões de métodos e atributos de um `DataFrame`. Experimente também `dados["inflacao_ipca_pct"].` para uma `Series`.
- **`dir(dados)`:** lista os nomes acessíveis no objeto, incluindo métodos e atributos. Para conhecer os nomes da própria biblioteca, use `dir(pd)`.
- **`help(dados.pct_change)`:** mostra a documentação de um método, seus parâmetros e exemplos. No Colab, também é possível escrever `dados.pct_change?` em uma célula.
- **Documentação oficial:** consulte a [referência da API do Pandas](https://pandas.pydata.org/docs/reference/index.html) para pesquisar métodos de `DataFrame` e `Series`.

**Qual é a diferença entre método e atributo?** Um método executa uma operação e é chamado com parênteses, como `dados.head()`. Um atributo fornece uma informação, como `dados.columns` ou `dados.shape` (sem parênteses).

**Pergunta para a turma:** se vocês precisassem descobrir como ordenar os dados por ano, qual dessas ferramentas usariam para encontrar o método adequado?

In [ ]:
# dir() também lista atributos; vamos selecionar apenas métodos públicos.
metodos_dataframe = sorted(
    nome for nome in dir(dados)
    if not nome.startswith("_") and callable(getattr(dados, nome))
)

print(f"Foram encontrados {len(metodos_dataframe)} métodos públicos no DataFrame:")
print(metodos_dataframe)

# Para obter ajuda detalhada sobre um método específico, descomente:
# help(dados.sort_values)


## 7. Guardar um resultado — Python → Excel

A análise pode voltar para uma planilha, agora contendo apenas o período e as variáveis escolhidas.

Com `.to_excel()`, salvamos um `DataFrame` em Excel. Usamos `index=False` para **não criar uma coluna extra** com o índice do Pandas.

In [ ]:
resumo_recente = dados.loc[
    dados["ano"] >= 2021,
    ["ano", "pib_crescimento_real_pct", "desemprego_pct", "inflacao_ipca_pct"]
]

resumo_recente.to_excel("resumo_economia_brasileira_2021_2025.xlsx", index=False)
print("Resumo exportado para Excel.")
resumo_recente

### Exercício 6 — relatório econômico integrado

**Situação:** você precisa construir, para uma nota econômica, um resumo do Brasil entre **2021 e 2025**.

1. Crie um `DataFrame` apenas com esse período.
2. Exiba `ano`, `pib_crescimento_real_pct`, `desemprego_pct` e `inflacao_ipca_pct`.
3. Calcule a **inflação média anual** do período e a **taxa média anual de desemprego**.
4. Identifique quais anos apresentaram crescimento real do PIB **superior a 3%**.
5. Exiba os resultados e escreva uma **interpretação econômica breve**, com base nas informações encontradas.

**Opcional:** exporte para Excel a tabela do período.

In [ ]:
# Crie a tabela apenas para 2021 a 2025.
# Selecione as quatro colunas pedidas.
# Calcule as médias de inflação e desemprego nesse período.
# Filtre os anos com crescimento real > 3%.
# Mostre resultados com print() e explique o que encontrou.

## 8. Fechamento — do arquivo a uma pergunta econômica

| Nova ferramenta | Para que serve? |
|---|---|
| `pd.read_excel()` | Importar uma planilha para Python |
| `head()` / `tail()` | Visualizar amostras dos dados |
| `shape` / `columns` / `info()` | Conhecer a estrutura da base |
| `dados["coluna"]` | Selecionar uma variável |
| `dados[["A", "B"]]` | Selecionar várias variáveis |
| `dados[condição]` / `.loc` | Filtrar anos e recuperar linhas |
| `.mean()` / `.max()` / `.min()` | Produzir estatísticas |
| `def` / `return` / `for` | Construir uma função própria para taxas de variação |
| `.pct_change()` | Obter o mesmo resultado diretamente em uma série do Pandas (multiplicar por 100 para expressar em %) |
| `dir(dados)` / `help(...)` | Descobrir métodos e consultar sua documentação |
| `.to_excel()` | Exportar resultados para uma planilha |



## 9. Exercício extra

**Pergunta:** em quais anos tivemos **inflação acima de 6%** **e** **desemprego acima de 10%**?

- Use `&` para combinar as duas condições.
- Exiba apenas `Data`, `IPCA` e `Desemprego`.
- Compare com os anos de crescimento real negativo do PIB e comente se as duas seleções são iguais.



In [ ]:
# Combine inflação > 6 e desemprego > 10.
# Exiba as três colunas escolhidas.
# Compare com a tabela de anos de crescimento real negativo.

## 10. Bibliografia e fontes

**Pandas — documentação oficial**

- [Começando com Pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Leitura de planilhas: `pandas.read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html)
- [Indexação e seleção de dados](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [Variação percentual: `Series.pct_change`](https://pandas.pydata.org/docs/reference/api/pandas.Series.pct_change.html)
- [Referência da API: métodos e atributos de DataFrame e Series](https://pandas.pydata.org/docs/reference/index.html)
- [Exportação: `DataFrame.to_excel`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html)

**Python — retomada**

- DOWNEY, Allen B. *Think Python*, 3ª ed., 2024. [Livro gratuito](https://allendowney.github.io/ThinkPython/).

**Fontes dos dados econômicos:** consultar a aba `Fontes_e_metodos` do arquivo Excel (IBGE Contas Nacionais, PNAD Contínua, IPCA e série SGS do Banco Central). A tabela reúne observações anuais de 2012 a 2025. Dados históricos podem sofrer revisões.

## 11. Gabarito — consultar depois de tentar

As respostas são **possibilidades de implementação**. Métodos equivalentes são válidos quando respondem à pergunta corretamente.

<details>
<summary>Exercício 1 — conhecer a base — abrir solução</summary>

```python
print(dados.head(4))
print(dados.tail(2))
print(dados.shape)  # (14, 5) antes de acrescentar a nova coluna
print(dados["ano"].iloc[0], dados["ano"].iloc[-1])  # 2012 e 2025
```

**Observação:** depois de criar a coluna de crescimento nominal, `shape` passa a indicar **6 colunas**.

</details>

<details>
<summary>Exercício 2 — inflação e crescimento real — abrir solução</summary>

```python
print(dados[["ano", "inflacao_ipca_pct"]].tail(5))
print(dados[["ano", "pib_crescimento_real_pct"]].tail(5))
```

De 2021 a 2025, o crescimento real do PIB registrado na planilha é positivo em todos os anos.

</details>

<details>
<summary>Exercício 3 — identificar períodos — abrir solução</summary>

```python
alto_desemprego = dados[dados["desemprego_pct"] > 10]
print(alto_desemprego[["ano", "desemprego_pct"]])

inflacao_recente = dados[
    (dados["ano"] >= 2020) &
    (dados["inflacao_ipca_pct"] > 5)
]
print(inflacao_recente[["ano", "inflacao_ipca_pct"]])
```

Com o segundo filtro, surgem os anos **2021 e 2022**.

</details>

<details>
<summary>Exercício 4 — desemprego e inflação — abrir solução</summary>

```python
print("Média do desemprego:", dados["desemprego_pct"].mean())
print("Maior desemprego:", dados["desemprego_pct"].max())

indice = dados["desemprego_pct"].idxmax()
print(dados.loc[indice, ["ano", "desemprego_pct"]])

indice_ipca = dados["inflacao_ipca_pct"].idxmax()
print(dados.loc[indice_ipca, ["ano", "inflacao_ipca_pct"]])
```

No conjunto de dados, o maior desemprego anual ocorre em **2021**, e o maior IPCA anual ocorre em **2015**.

</details>

<details>
<summary>Exercício 5 — criar uma função para calcular variações percentuais — abrir solução</summary>

```python
# Parte A: função construída por nós.
def calcular_variacao_pct(valor_anterior, valor_atual):
    return (valor_atual / valor_anterior - 1) * 100

print(f"{calcular_variacao_pct(100, 110):.2f}%")  # 10.00%

# Parte B: repetir o cálculo ao longo da série do PIB nominal.
pib_nominal = dados["pib_nominal_bilhoes_rs"].tolist()
taxas_manuais = [float("nan")]

for i in range(1, len(pib_nominal)):
    taxa = calcular_variacao_pct(pib_nominal[i - 1], pib_nominal[i])
    taxas_manuais.append(taxa)

dados["crescimento_nominal_funcao_pct"] = taxas_manuais
print(dados[["ano", "pib_nominal_bilhoes_rs", "crescimento_nominal_funcao_pct"]].head())

# Conferir, após executar a célula que apresenta pct_change():
print(dados[["ano", "crescimento_nominal_funcao_pct", "pib_nominal_crescimento_pct"]].head())
```

No primeiro ano, os dois métodos produzem um valor ausente. Nos demais, calculam a mesma variação percentual, multiplicando a proporção por 100.

</details>

<details>
<summary>Exercício 6 — relatório integrado — abrir solução</summary>

```python
periodo = dados[dados["ano"] >= 2021]

colunas = ["ano", "pib_crescimento_real_pct", "desemprego_pct", "inflacao_ipca_pct"]
print(periodo[colunas])

media_inflacao = periodo["inflacao_ipca_pct"].mean()
media_desemprego = periodo["desemprego_pct"].mean()

print(f"Inflação média anual: {media_inflacao:.2f}%")
print(f"Desemprego médio anual: {media_desemprego:.2f}%")

anos_crescimento = periodo[periodo["pib_crescimento_real_pct"] > 3]
print(anos_crescimento[["ano", "pib_crescimento_real_pct"]])

periodo[colunas].to_excel("relatorio_2021_2025.xlsx", index=False)
```

No recorte, o crescimento real supera 3% em **2021, 2022, 2023 e 2024**. As taxas anuais de desemprego diminuem entre 2021 e 2025.

</details>

<details>
<summary>Exercício extra — inflação e desemprego altos — abrir solução</summary>

```python
anos_criticos = dados[
    (dados["inflacao_ipca_pct"] > 6) &
    (dados["desemprego_pct"] > 10)
]
print(anos_criticos[["ano", "inflacao_ipca_pct", "desemprego_pct"]])

recessoes = dados[dados["pib_crescimento_real_pct"] < 0]
print(recessoes[["ano", "pib_crescimento_real_pct"]])
```

- Inflação acima de 6% **e** desemprego acima de 10%: **2016 e 2021**.
- Crescimento real negativo: **2015, 2016 e 2020**.

Os grupos não coincidem: a coincidência de indicadores deve ser interpretada, não tomada como prova causal.

</details>